# ValoStats — Notebook 1: construcción de perfiles de juego

## 1. Planteamiento del problema

ValoStats nace porque las estadísticas competitivas de Valorant suelen estar disponibles de forma separada, pero no siempre entregan una interpretación clara del rendimiento real de un jugador.

Métricas como ACS, K/D, ADR, KAST, porcentaje de headshots, first kills y winrate pueden indicar aspectos distintos del juego, pero por sí solas no responden completamente preguntas como:

- ¿El jugador está rindiendo bien?
- ¿Qué estilo de juego tiene?
- ¿Está mejorando o bajando su rendimiento?
- ¿Cómo se compara con jugadores de lobbies similares?

Por esto, el problema que busca resolver ValoStats es transformar estadísticas competitivas recientes en una lectura clara del jugador. En este notebook se aborda específicamente la construcción de perfiles de estilo de juego.

## 2. Objetivo de este notebook

El objetivo de este notebook es construir y validar perfiles de jugadores usando clustering.

En concreto, se busca:

1. Preparar variables que representen dimensiones del estilo de juego.
2. Evaluar distintos valores de `k` para K-Means.
3. Elegir una cantidad de perfiles interpretable.
4. Entrenar K-Means con `k = 3`.
5. Interpretar los clusters y asignar nombres a los perfiles.
6. Entrenar un clasificador que permita asignar nuevas observaciones a esos perfiles.

## 3. Importación de librerías y rutas

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_project_root():
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        has_data = (candidate / "data" / "val_stats.csv").exists()
        has_src = (candidate / "src").exists()

        if has_data and has_src:
            return candidate

    if (current.parent / "data" / "val_stats.csv").exists():
        return current.parent

    return current

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data" / "val_stats.csv"

print("Raíz del proyecto:", PROJECT_ROOT)
print("Dataset:", DATA_PATH)

## 4. Carga y preparación de datos

In [ ]:
from src.preprocessing import prepare_pipeline

raw_df, clean_df, feature_df, cluster_df, scaled_df, scaler = prepare_pipeline(DATA_PATH)

print("Filas originales:", len(raw_df))
print("Filas limpias:", len(clean_df))
print("Variables para clustering:", feature_df.shape[1])

In [ ]:
raw_df.head()

In [ ]:
feature_df.describe().T

## 5. Variables usadas para construir perfiles

Para construir los perfiles se utilizan variables derivadas del dataset original. Estas variables buscan representar dimensiones del estilo de juego y no solo estadísticas aisladas.

Las dimensiones principales son:

| Variable | Interpretación |
|---|---|
| agresividad | Capacidad de generar kills respecto a muertes o participación ofensiva. |
| precision | Precisión mecánica, principalmente asociada al porcentaje de headshots. |
| impacto | Aporte ofensivo en situaciones relevantes, usando métricas como clutches, aces o first bloods según disponibilidad. |
| soporte | Aporte mediante asistencias y participación de apoyo. |
| eficiencia | Relación entre rendimiento y resultados. |
| entry_power | Participación o efectividad en duelos iniciales. |
| consistencia | Estabilidad del jugador en sus métricas principales. |

Antes de aplicar clustering, estas variables se escalan con `StandardScaler`. Esto evita que una métrica con mayor escala numérica domine el agrupamiento.

## 6. Evaluación de distintos valores de k

Para elegir la cantidad de clusters se evalúan distintos valores de `k`.

Se consideran principalmente:

- Silhouette score: mide separación y cohesión de los grupos. Valores más altos suelen ser mejores.
- Davies-Bouldin: mide separación entre clusters. Valores más bajos suelen ser mejores.
- Inertia o método del codo: ayuda a observar desde qué punto agregar más clusters entrega poca mejora.

La elección final no depende solo de la métrica, sino también de la interpretabilidad de los perfiles.

In [ ]:
from src.clustering import evaluate_kmeans_range
from src.visualization import plot_k_metrics, plot_elbow

k_results = evaluate_kmeans_range(
    scaled_df,
    k_min=2,
    k_max=8
)

k_results

In [ ]:
plot_k_metrics(k_results)

In [ ]:
plot_elbow(k_results)

## 7. Elección de k

Se selecciona `k = 3` porque permite obtener perfiles diferenciables e interpretables dentro del contexto del juego.

Con menos clusters se pierde información sobre diferencias de estilo. Con más clusters, los grupos pueden volverse demasiado específicos y difíciles de explicar.

Con `k = 3`, los perfiles se interpretan como:

- Alto impacto
- Apoyo táctico
- Ofensivo consistente

Esta elección responde tanto a las métricas de clustering como a la necesidad de que los grupos tengan sentido para un usuario de Valorant.

## 8. Entrenamiento de K-Means con k = 3

In [ ]:
from src.clustering import train_kmeans

kmeans_model, clusters, clustering_metrics = train_kmeans(
    scaled_df,
    n_clusters=3
)

clustering_metrics

## 9. Visualización con PCA

PCA se utiliza solo como herramienta de visualización. Permite proyectar los datos en dos dimensiones para observar la distribución de los clusters.

Esta visualización no reemplaza la validación cuantitativa, pero ayuda a explicar de forma más clara cómo se distribuyen los jugadores en el espacio de perfiles.

In [ ]:
from src.clustering import apply_pca

feature_df["cluster"] = clusters

pca_model, pca_result = apply_pca(scaled_df)

feature_df["pca_1"] = pca_result[:, 0]
feature_df["pca_2"] = pca_result[:, 1]

feature_df[["name", "cluster", "pca_1", "pca_2"]].head()

## 10. Interpretación de clusters y asignación de perfiles

Luego de entrenar K-Means, se revisan los promedios de cada cluster para interpretar qué representa cada grupo.

A partir de esos promedios, los clusters se renombran como perfiles de juego:

- Alto impacto: jugadores con mayor aporte ofensivo o impacto en rondas.
- Apoyo táctico: jugadores con más peso en soporte, asistencias o participación de apoyo.
- Ofensivo consistente: jugadores con rendimiento ofensivo estable y menos extremo.

In [ ]:
from src.clustering import (
    assign_player_types,
    calculate_centroid_distances,
    add_secondary_profiles,
    build_cluster_summary,
)

feature_df, cluster_names, raw_cluster_summary = assign_player_types(feature_df)

cluster_names

In [ ]:
raw_cluster_summary

In [ ]:
distance_df = calculate_centroid_distances(kmeans_model, scaled_df)

feature_df = add_secondary_profiles(
    feature_df,
    distance_df,
    cluster_names
)

feature_df[
    [
        "name",
        "player_type",
        "secondary_profile",
        "profile_mix"
    ]
].head(10)

## 11. Visualización de perfiles

In [ ]:
from src.visualization import plot_pca_clusters, plot_radar_profiles

plot_pca_clusters(feature_df)

In [ ]:
radar_fig, profile_summary = plot_radar_profiles(feature_df)

radar_fig

In [ ]:
profile_summary

## 12. Validación e interpretación de perfiles

Los perfiles se validan de tres formas:

1. Métricas internas de clustering:
   - Silhouette score.
   - Davies-Bouldin.
   - Inertia.

2. Interpretabilidad:
   - Se revisan los promedios internos de cada cluster.
   - Se verifica si cada grupo tiene una identidad clara.

3. Aplicabilidad:
   - Se entrena un clasificador supervisado usando las etiquetas generadas por K-Means.

In [ ]:
print("Métricas de clustering con k=3")

for metric, value in clustering_metrics.items():
    print(f"{metric}: {value}")

## 13. Clasificador de perfiles

In [ ]:
from src.classification import train_random_forest

X = scaled_df
y = feature_df["player_type"]

rf_model, X_train, X_test, y_train, y_test, y_pred, classification_metrics = train_random_forest(
    X,
    y
)

classification_metrics["accuracy"]

In [ ]:
print(classification_metrics["classification_report"])

## 14. Recomendación asociada al perfil

Una vez asignado el perfil, se puede generar una recomendación básica asociada al estilo de juego.

Esta recomendación no reemplaza al sistema de jugadores similares del segundo notebook, pero sirve como primera interpretación del perfil.

In [ ]:
from src.recommendation import generate_recommendation

feature_df["recommendation"] = feature_df.apply(
    generate_recommendation,
    axis=1
)

feature_df[
    [
        "name",
        "player_type",
        "secondary_profile",
        "recommendation"
    ]
].head(10)